# Aprendizado por Reforço com SARSA — Labirinto 50×50

## Cenário

Um agente robô é colocado no canto superior esquerdo de um labirinto **50×50**. Seu objetivo é encontrar a saída no canto inferior direito. O labirinto tem **paredes intransponíveis** e **corredores livres**. O agente não conhece o mapa — ele precisa **explorar e aprender** pelo algoritmo SARSA.

---

## Recapitulando: O que é SARSA?

SARSA é um algoritmo **on-policy**: o agente aprende *com as ações que realmente executa*, inclusive as aleatórias (exploração).

**Equação de atualização:**

$$Q(s, a) \leftarrow Q(s, a) + \alpha \Big[ r + \gamma \cdot Q(s', a') - Q(s, a) \Big]$$

A diferença crítica em relação ao Q-Learning: **`a'` é a ação que o agente vai de fato tomar** no próximo estado — não o máximo hipotético.

---

## Estrutura do problema

| Elemento | Definição |
|---|---|
| **Estado** | Posição `(linha, coluna)` do agente no labirinto |
| **Ações** | Cima, Baixo, Esquerda, Direita (4 ações) |
| **Recompensa** | +100 ao chegar na meta, −1 por passo, −5 ao bater em parede |
| **Meta** | Posição (48, 48) |
| **Início** | Posição (0, 0) |

> **Reward Shaping:** Para acelerar a convergência em labirintos grandes, adicionamos um pequeno bônus positivo (+0.3) quando o agente se aproxima da meta e uma penalidade (−0.3) quando se afasta. Isso **não altera a política ótima**, apenas acelera o aprendizado.

## 1. Importações e Geração do Labirinto

O labirinto é gerado com o algoritmo **DFS Iterativo** (*Depth-First Search* com backtracking), que garante:
- Sempre existe exatamente **um caminho** entre dois pontos quaisquer
- O labirinto é **perfeito** (sem ilhas ou regiões inacessíveis)
- O resultado é **reproduzível** pela seed

**Representação:** `0` = célula livre | `1` = parede

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import random
import warnings
warnings.filterwarnings('ignore')

# Reprodutibilidade
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

MAZE_SIZE = 50
START     = (0, 0)
GOAL      = (48, 48)   # Célula livre mais próxima do canto (índice par = sala DFS)


def generate_maze(size: int = 50, seed: int = 42) -> np.ndarray:
    """
    Gera um labirinto perfeito via DFS iterativo.
    Salas ficam em índices pares; paredes esculpidas em índices ímpares.
    """
    random.seed(seed)
    maze = np.ones((size, size), dtype=np.int8)   # Tudo começa como parede
    visited = set()

    stack = [START]
    visited.add(START)
    maze[START] = 0   # Abre a célula inicial

    while stack:
        r, c = stack[-1]
        # Vizinhos a 2 passos (outras salas)
        vizinhos = [(r-2, c), (r+2, c), (r, c-2), (r, c+2)]
        random.shuffle(vizinhos)

        avancou = False
        for nr, nc in vizinhos:
            if 0 <= nr < size and 0 <= nc < size and (nr, nc) not in visited:
                # Esculpe a parede entre a sala atual e a vizinha
                maze[(r + nr) // 2, (c + nc) // 2] = 0
                maze[nr, nc] = 0
                visited.add((nr, nc))
                stack.append((nr, nc))
                avancou = True
                break

        if not avancou:
            stack.pop()   # Backtrack

    return maze


maze = generate_maze(MAZE_SIZE, SEED)

print(f"Labirinto {MAZE_SIZE}×{MAZE_SIZE} gerado com sucesso.")
print(f"  Células livres : {(maze == 0).sum():>5}")
print(f"  Paredes        : {(maze == 1).sum():>5}")
print(f"  Início: {START}  |  Meta: {GOAL}")
print(f"  maze[START] = {maze[START]} (deve ser 0)")
print(f"  maze[GOAL]  = {maze[GOAL]}  (deve ser 0)")

## 2. Visualização do Labirinto (antes do treinamento)

O mapa completo do labirinto. O agente **não tem acesso a esta visão** — ele aprende apenas pelas recompensas recebidas a cada passo.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

ax.imshow(maze, cmap='binary', interpolation='nearest', origin='upper')

# Marca início e meta
ax.scatter(START[1], START[0], s=180, c='limegreen', zorder=10,
           edgecolors='black', linewidth=1.5, marker='o', label='Início (0,0)')
ax.scatter(GOAL[1],  GOAL[0],  s=250, c='crimson',   zorder=10,
           edgecolors='black', linewidth=1.5, marker='*', label='Meta (48,48)')

ax.set_title('Labirinto 50×50 — Antes do Treinamento', fontsize=14, pad=12)
ax.set_xticks([])
ax.set_yticks([])
ax.legend(loc='lower right', fontsize=11, framealpha=0.9)
plt.tight_layout()
plt.show()

## 3. Modelagem do Ambiente

Para o SARSA, precisamos codificar o labirinto como um **Processo de Decisão de Markov (MDP)**:

- **Estado → índice escalar:** A posição `(r, c)` é convertida para um único inteiro: `idx = r × 50 + c`
  - Isso simplifica o acesso à Q-table (sem precisar de índice 2D)
- **4 ações disponíveis:** Cima (0), Baixo (1), Esquerda (2), Direita (3)
- **Transição:** Se a célula destino for livre (`maze == 0`), o agente se move. Caso contrário, **permanece no lugar** e recebe penalidade de parede.
- **Reward shaping:** Bônus proporcional à redução na distância de Manhattan até a meta.

In [ ]:
NUM_STATES  = MAZE_SIZE * MAZE_SIZE   # 2500 estados
NUM_ACTIONS = 4                        # Cima, Baixo, Esquerda, Direita

# Deltas (dr, dc) para cada ação
DELTAS = {
    0: (-1,  0),   # Cima
    1: ( 1,  0),   # Baixo
    2: ( 0, -1),   # Esquerda
    3: ( 0,  1),   # Direita
}
ACTION_NAMES = {0: 'Cima', 1: 'Baixo', 2: 'Esquerda', 3: 'Direita'}

def pos_to_idx(r: int, c: int) -> int:
    return r * MAZE_SIZE + c

def idx_to_pos(idx: int):
    return idx // MAZE_SIZE, idx % MAZE_SIZE

def manhattan(r1, c1, r2, c2) -> float:
    return abs(r1 - r2) + abs(c1 - c2)


def executar_acao(r: int, c: int, action: int, shaping_weight: float = 0.3):
    """
    Executa a ação no ambiente.
    Retorna (next_r, next_c, reward).
    """
    dr, dc   = DELTAS[action]
    nr, nc   = r + dr, c + dc
    goal_r, goal_c = GOAL

    # Movimento válido?
    if 0 <= nr < MAZE_SIZE and 0 <= nc < MAZE_SIZE and maze[nr, nc] == 0:
        if (nr, nc) == GOAL:
            return nr, nc, 100.0          # Recompensa terminal
        # Reward shaping: incentiva aproximação à meta
        dist_antes = manhattan(r,  c,  goal_r, goal_c)
        dist_depois = manhattan(nr, nc, goal_r, goal_c)
        shaping = shaping_weight * (dist_antes - dist_depois)
        return nr, nc, -1.0 + shaping     # Custo de passo + shaping
    else:
        return r, c, -5.0                  # Penalidade de parede (agente não se move)


def escolher_acao(state_idx: int, q_table: np.ndarray, eps: float) -> int:
    """Política epsilon-greedy."""
    if np.random.uniform() < eps:
        return np.random.randint(NUM_ACTIONS)   # Exploração
    return int(np.argmax(q_table[state_idx]))    # Exploração do melhor Q


print(f"Espaço de estados : {NUM_STATES}")
print(f"Espaço de ações   : {NUM_ACTIONS} — {list(ACTION_NAMES.values())}")
print(f"Tamanho da Q-table: {NUM_STATES} × {NUM_ACTIONS} = {NUM_STATES * NUM_ACTIONS} valores")

## 4. Inicialização da Q-Table e Hiperparâmetros

A Q-table tem dimensão **2.500 × 4** — uma linha por posição no labirinto, uma coluna por ação.

**Epsilon decay:** `ε` começa em 1.0 (100% exploração aleatória) e cai gradualmente até 0.01. Isso simula a transição natural de *"não sei nada, exploro tudo"* para *"aprendi o suficiente, confio na minha política"*.

| Hiperparâmetro | Valor | Justificativa |
|---|---|---|
| `learning_rate (α)` | 0.15 | Aprendizado um pouco mais rápido para espaço grande |
| `discount_factor (γ)` | 0.99 | Recompensas futuras muito importantes (caminho longo) |
| `epsilon` inicial | 1.0 | Total exploração no início |
| `epsilon_decay` | 0.998 | Decaimento lento — labirinto grande exige longa exploração |
| `num_episodes` | 3.000 | Suficiente para convergência com reward shaping |
| `max_steps` | 3.000 | Limite de passos por episódio |

In [ ]:
# --- Hiperparâmetros ---
learning_rate   = 0.15
discount_factor = 0.99
epsilon         = 1.0
epsilon_min     = 0.01
epsilon_decay   = 0.998
num_episodes    = 3000
max_steps       = 3000

# --- Q-table inicializada com zeros ---
q_table = np.zeros((NUM_STATES, NUM_ACTIONS))

print(f"Q-table shape: {q_table.shape}")
print(f"Epsilon após 1000 ep : {epsilon * epsilon_decay**1000:.4f}")
print(f"Epsilon após 2000 ep : {epsilon * epsilon_decay**2000:.4f}")
print(f"Epsilon após 3000 ep : {epsilon * epsilon_decay**3000:.6f}")

## 5. Loop de Treinamento SARSA

### Fluxo por episódio:

```
1. Inicializa estado s = START
2. Escolhe ação a via ε-greedy          ← SARSA escolhe ANTES do loop
3. LOOP:
   a. Executa ação a → obtém (s', r)
   b. Escolhe próxima ação a' via ε-greedy em s'
   c. Atualiza: Q(s,a) ← Q(s,a) + α[r + γ·Q(s',a') - Q(s,a)]
   d. s ← s', a ← a'
   e. Se s == META: encerra episódio
4. Decai epsilon
```

> **Ponto crítico do SARSA:** O `Q(s', a')` na equação usa a ação `a'` que *já foi sorteada* pela política ε-greedy. Se `a'` foi aleatória (exploração), o agente aprende com essa escolha subótima — sendo mais conservador do que o Q-Learning.

---
_Tempo estimado de execução: 1–3 minutos dependendo do hardware._

In [ ]:
recompensas_episodio = []
passos_episodio      = []
metas_atingidas      = 0
visit_freq           = np.zeros((MAZE_SIZE, MAZE_SIZE), dtype=np.int32)  # Frequência de visitas

for episode in range(num_episodes):

    # --- Inicialização do episódio ---
    r, c       = START
    state_idx  = pos_to_idx(r, c)
    action     = escolher_acao(state_idx, q_table, epsilon)   # SARSA: primeira ação ANTES do while
    total_reward = 0.0
    chegou     = False

    for step in range(max_steps):

        # 1. Executa ação, observa resultado
        nr, nc, reward = executar_acao(r, c, action)
        next_idx       = pos_to_idx(nr, nc)
        total_reward  += reward
        visit_freq[nr, nc] += 1

        # 2. Escolhe próxima ação (on-policy: mesma política ε-greedy)
        next_action = escolher_acao(next_idx, q_table, epsilon)

        # 3. Atualização SARSA
        td_error = reward + discount_factor * q_table[next_idx, next_action] - q_table[state_idx, action]
        q_table[state_idx, action] += learning_rate * td_error

        # 4. Transição de estado
        r, c       = nr, nc
        state_idx  = next_idx
        action     = next_action

        # 5. Condição de término
        if (r, c) == GOAL:
            metas_atingidas += 1
            chegou = True
            break

    # Decaimento de epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    recompensas_episodio.append(total_reward)
    passos_episodio.append(step + 1)

    # Log a cada 500 episódios
    if (episode + 1) % 500 == 0:
        taxa = 100 * metas_atingidas / (episode + 1)
        print(f"Ep {episode+1:>5} | ε={epsilon:.4f} | "
              f"Recompensa média (últ. 100): {np.mean(recompensas_episodio[-100:]):>8.1f} | "
              f"Taxa de sucesso: {taxa:.1f}%")

print(f"\nTreinamento concluído.")
print(f"Meta atingida em {metas_atingidas}/{num_episodes} episódios ({100*metas_atingidas/num_episodes:.1f}%)")

## 6. Extração do Caminho Aprendido (Política Greedy)

Com o treinamento concluído, executamos o agente **sem exploração** (ε = 0): ele sempre escolhe a ação com maior Q-value. Isso revela o caminho que a política aprendida considera ótimo.

In [ ]:
def extrair_caminho_greedy(q_table: np.ndarray, limite: int = 10000):
    """Executa a política greedy e retorna o caminho percorrido."""
    r, c   = START
    caminho = [(r, c)]
    visitados = {(r, c)}

    for _ in range(limite):
        idx    = pos_to_idx(r, c)
        action = int(np.argmax(q_table[idx]))
        dr, dc = DELTAS[action]
        nr, nc = r + dr, c + dc

        # Move apenas se válido
        if 0 <= nr < MAZE_SIZE and 0 <= nc < MAZE_SIZE and maze[nr, nc] == 0:
            r, c = nr, nc

        if (r, c) in visitados:
            print("  Aviso: loop detectado — política não convergiu completamente.")
            break

        caminho.append((r, c))
        visitados.add((r, c))

        if (r, c) == GOAL:
            break

    return caminho


caminho = extrair_caminho_greedy(q_table)
chegou_meta = caminho[-1] == GOAL

print(f"Ponto final       : {caminho[-1]}")
print(f"Meta atingida     : {'SIM' if chegou_meta else 'NÃO'}")
print(f"Comprimento do caminho: {len(caminho)} passos")

## 7. Visualização Principal — Caminho no Labirinto

O gradiente de cores do caminho vai de **amarelo** (início) a **roxo escuro** (fim), mostrando a progressão espacial do agente pelo labirinto.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 13))

# --- Fundo: labirinto ---
ax.imshow(maze, cmap='binary', interpolation='nearest', origin='upper', zorder=1)

# --- Caminho com gradiente de cor ---
if len(caminho) > 1:
    pontos    = np.array([[c, r] for r, c in caminho], dtype=float)   # (x=col, y=row)
    segmentos = np.stack([pontos[:-1], pontos[1:]], axis=1)
    n_seg     = len(segmentos)

    # Gradiente plasma: amarelo → laranja → roxo
    cores = plt.cm.plasma(np.linspace(0.05, 0.95, n_seg))
    lc    = LineCollection(segmentos, colors=cores, linewidths=2.0,
                           alpha=0.92, zorder=5, capstyle='round')
    ax.add_collection(lc)

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap='plasma',
                                norm=plt.Normalize(vmin=0, vmax=len(caminho)))
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.55, pad=0.02)
    cbar.set_label('Número do passo', fontsize=12)

# --- Marcadores ---
ax.scatter(START[1], START[0], s=300, c='limegreen', zorder=10,
           edgecolors='black', linewidth=2, marker='o')
ax.scatter(GOAL[1],  GOAL[0],  s=400, c='crimson',   zorder=10,
           edgecolors='black', linewidth=2, marker='*')

# Anotações de texto
ax.annotate('INÍCIO', xy=(START[1], START[0]), xytext=(START[1]+1.5, START[0]+2.5),
            fontsize=9, color='white', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='green', alpha=0.8))
ax.annotate('META', xy=(GOAL[1], GOAL[0]), xytext=(GOAL[1]-7, GOAL[0]-2),
            fontsize=9, color='white', fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='crimson', alpha=0.8))

# --- Títulos e formatação ---
status = f"Meta atingida em {len(caminho)} passos" if chegou_meta else f"Meta não atingida ({len(caminho)} passos)"
ax.set_title(f'SARSA — Labirinto 50×50\n{status}', fontsize=15, pad=14, fontweight='bold')
ax.set_xticks([])
ax.set_yticks([])

# Legenda manual
leg = [
    mpatches.Patch(color='limegreen', label='Início (0,0)'),
    mpatches.Patch(color='crimson',   label='Meta (48,48)'),
    mpatches.Patch(color='gold',      label='Início do trajeto'),
    mpatches.Patch(color='indigo',    label='Fim do trajeto'),
]
ax.legend(handles=leg, loc='upper right', fontsize=10,
          framealpha=0.9, edgecolor='gray')

plt.tight_layout()
plt.savefig('labirinto_caminho_sarsa.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura salva como 'labirinto_caminho_sarsa.png'")

## 8. Curva de Aprendizado

Dois gráficos que mostram a evolução do aprendizado ao longo dos episódios:
- **Recompensa acumulada:** deve aumentar com o tempo (agente aprende a minimizar passos)
- **Passos por episódio:** deve diminuir com o tempo (agente aprende caminhos mais curtos)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
JANELA = 100

# --- Recompensa por episódio ---
ax1 = axes[0]
ax1.plot(recompensas_episodio, alpha=0.25, color='steelblue', linewidth=0.8)
media_recomp = np.convolve(recompensas_episodio, np.ones(JANELA)/JANELA, mode='valid')
ax1.plot(range(JANELA-1, num_episodes), media_recomp,
         color='steelblue', linewidth=2.5, label=f'Média móvel ({JANELA} ep.)')
ax1.set_xlabel('Episódio', fontsize=12)
ax1.set_ylabel('Recompensa total', fontsize=12)
ax1.set_title('Recompensa Acumulada por Episódio', fontsize=13)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# --- Passos por episódio ---
ax2 = axes[1]
ax2.plot(passos_episodio, alpha=0.25, color='tomato', linewidth=0.8)
media_passos = np.convolve(passos_episodio, np.ones(JANELA)/JANELA, mode='valid')
ax2.plot(range(JANELA-1, num_episodes), media_passos,
         color='tomato', linewidth=2.5, label=f'Média móvel ({JANELA} ep.)')
ax2.set_xlabel('Episódio', fontsize=12)
ax2.set_ylabel('Passos até encerrar', fontsize=12)
ax2.set_title('Passos por Episódio', fontsize=13)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.suptitle('Curva de Aprendizado — SARSA no Labirinto 50×50',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('curva_aprendizado_sarsa.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Mapa de Calor — Frequência de Visitas

Este mapa mostra **quais células foram mais visitadas durante todo o treinamento**. Células com alta frequência (mais quentes) indicam regiões exploradas intensamente — geralmente corredores de passagem obrigatória ou áreas de incerteza no início do aprendizado.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# --- Mapa de calor de visitas ---
ax1 = axes[0]
# Mascara as paredes para não aparecerem no heatmap
freq_mascarada = np.ma.masked_where(maze == 1, visit_freq)

im = ax1.imshow(freq_mascarada, cmap='YlOrRd', interpolation='nearest',
                origin='upper', aspect='equal')
# Sobrepõe o contorno das paredes
ax1.imshow(maze, cmap='binary', interpolation='nearest',
           origin='upper', alpha=0.35, zorder=2)

ax1.scatter(START[1], START[0], s=250, c='blue',   zorder=10, marker='o', label='Início')
ax1.scatter(GOAL[1],  GOAL[0],  s=350, c='white',  zorder=10, marker='*',
            edgecolors='black', linewidth=1.5, label='Meta')

cbar1 = fig.colorbar(im, ax=ax1, shrink=0.7)
cbar1.set_label('Frequência de visitas', fontsize=11)
ax1.set_title('Mapa de Calor — Visitas durante o Treinamento', fontsize=13)
ax1.set_xticks([])
ax1.set_yticks([])
ax1.legend(loc='upper right', fontsize=10, framealpha=0.85)

# --- Mapa de Q-values máximos ---
ax2 = axes[1]
max_q = np.max(q_table, axis=1).reshape(MAZE_SIZE, MAZE_SIZE)
max_q_mascarado = np.ma.masked_where(maze == 1, max_q)

im2 = ax2.imshow(max_q_mascarado, cmap='viridis', interpolation='nearest',
                  origin='upper', aspect='equal')
ax2.imshow(maze, cmap='binary', interpolation='nearest',
           origin='upper', alpha=0.35, zorder=2)

ax2.scatter(START[1], START[0], s=250, c='limegreen', zorder=10, marker='o', label='Início')
ax2.scatter(GOAL[1],  GOAL[0],  s=350, c='crimson',   zorder=10, marker='*',
            edgecolors='black', linewidth=1.5, label='Meta')

cbar2 = fig.colorbar(im2, ax=ax2, shrink=0.7)
cbar2.set_label('max Q(s, a)', fontsize=11)
ax2.set_title('Mapa de Q-values Máximos por Célula', fontsize=13)
ax2.set_xticks([])
ax2.set_yticks([])
ax2.legend(loc='upper right', fontsize=10, framealpha=0.85)

plt.suptitle('Análise do Aprendizado — SARSA no Labirinto 50×50',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('analise_aprendizado_sarsa.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Resumo e Interpretação Pedagógica

### O que observar nos gráficos:

| Gráfico | O que indica convergência |
|---|---|
| **Caminho no labirinto** | Trajeto contínuo de START → GOAL sem voltas desnecessárias |
| **Recompensa por episódio** | Curva crescente ao longo do tempo |
| **Passos por episódio** | Curva decrescente (menos passos = caminho mais eficiente) |
| **Mapa de calor** | Regiões quentes próximas ao caminho ótimo (agente aprendeu onde focar) |
| **Mapa de Q-values** | Gradiente de valores: alto perto da meta, baixo perto do início |

---

### Por que o SARSA é on-policy?

Quando o agente está explorando (ε > 0), ele às vezes toma ações ruins (bate em paredes). No SARSA, essas ações ruins **entram no cálculo do Q-value** via `Q(s', a')`. Isso faz o agente aprender uma política mais **realista e cautelosa**, ao contrário do Q-Learning que sempre simula que o próximo passo será perfeito.

---

### Para explorar mais:

- Troque o `SEED` e veja um labirinto diferente
- Reduza `num_episodes` para 500 e observe a degradação da política
- Remova o reward shaping e observe quantos episódios a mais são necessários
- Substitua a equação SARSA pela do Q-Learning (`np.max` em vez de `q_table[next_idx, next_action]`) e compare os caminhos

---

> **Referência:** Sutton & Barto, *Reinforcement Learning: An Introduction* (2nd ed.), Cap. 6.4 — [http://incompleteideas.net/book/the-book.html](http://incompleteideas.net/book/the-book.html)